In [ ]:
# Imports & Global Setup

import os
import torch
import pygame
import numpy as np
import torch.nn as nn
import gymnasium as gym
import torch.optim as optim
from collections import deque
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# gc.collect()
torch.cuda.empty_cache()
os.environ['CUDA_LAUNCH_BLOCKING'] = '1' 

# Seed everything
seed = 21
np.random.seed(seed)
np.random.default_rng(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
# Replay Memory

class ReplayMemory:
    def __init__(self, capacity):
        self.capacity = capacity
        
        self.states       = deque(maxlen=capacity)
        self.actions      = deque(maxlen=capacity)
        self.next_states  = deque(maxlen=capacity)
        self.rewards      = deque(maxlen=capacity)
        self.dones        = deque(maxlen=capacity)
        
    def store(self, state, action, next_state, reward, done):
        self.states.append(state)
        self.actions.append(action)
        self.next_states.append(next_state)
        self.rewards.append(reward)
        self.dones.append(done)
        
    def sample(self, batch_size):
        # np.random.seed(seed)  

        indices = np.random.choice(len(self), size=batch_size, replace=False)

        states = torch.stack([torch.as_tensor(self.states[i], dtype=torch.float32, device=device) for i in indices])
        actions = torch.as_tensor([self.actions[i] for i in indices], dtype=torch.long, device=device)
        next_states = torch.stack([torch.as_tensor(self.next_states[i], dtype=torch.float32, device=device) for i in indices])
        rewards = torch.as_tensor([self.rewards[i] for i in indices], dtype=torch.float32, device=device)
        dones = torch.as_tensor([self.dones[i] for i in indices], dtype=torch.bool, device=device)

        return states, actions, next_states, rewards, dones
    
    def __len__(self):
        return len(self.dones)


In [ ]:
# DQN Neural Network
class DQN_Network(nn.Module):
    def __init__(self, num_actions, input_dim):
        super(DQN_Network, self).__init__()

        self.FC = nn.Sequential(
            nn.Linear(input_dim, 12),
            nn.ReLU(inplace=True),
            nn.Linear(12, 8),
            nn.ReLU(inplace=True),
            nn.Linear(8, num_actions),
        )

        for layer in [self.FC]:
            for module in layer:
                if isinstance(module, nn.Linear):
                    nn.init.kaiming_uniform_(module.weight, nonlinearity="relu")
                    nn.init.constant_(module.bias, 0.0)  # Zeros out biases

    def forward(self, x):
        return self.FC(x)

In [ ]:
# DQN Agent

class DQN_Agent:
    def __init__(
        self,
        env,
        epsilon_max,
        epsilon_min,
        epsilon_decay,
        clip_grad_norm,
        learning_rate,
        discount,
        memory_capacity,
    ):

        self.loss_history = []
        self.running_loss = 0
        self.learned_counts = 0

        self.epsilon_max = epsilon_max
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.discount = discount

        self.action_space = env.action_space
        self.action_space.seed(seed)
        self.observation_space = env.observation_space
        self.replay_memory = ReplayMemory(memory_capacity)

        input_dim = self.observation_space.shape[0]
        output_dim = self.action_space.n

        self.main_network = DQN_Network(output_dim, input_dim).to(device)
        self.target_network = DQN_Network(output_dim, input_dim).to(device).eval()
        self.target_network.load_state_dict(self.main_network.state_dict())

        self.clip_grad_norm = clip_grad_norm

        # Huber loss 
        self.criterion = nn.SmoothL1Loss()

        self.optimizer = optim.Adam(self.main_network.parameters(), lr=learning_rate)

    def select_action(self, state):
        if np.random.random() < self.epsilon_max:
            return self.action_space.sample()

        if not torch.is_tensor(state):
            state = torch.as_tensor(state, dtype=torch.float32, device=device)

        with torch.no_grad():
            Q_values = self.main_network(state)
            return torch.argmax(Q_values).item()

    def learn(self, batch_size, done):
        states, actions, next_states, rewards, dones = self.replay_memory.sample(batch_size)

        actions = actions.unsqueeze(1)
        rewards = rewards.unsqueeze(1)
        dones = dones.unsqueeze(1)

        # Q(s,a) from main network
        predicted_q = self.main_network(states).gather(1, actions)

        with torch.no_grad():
            # Double-DQN
            next_actions = self.main_network(next_states).argmax(dim=1, keepdim=True)
            next_q_values = self.target_network(next_states).gather(1, next_actions)

            next_q_values[dones] = 0

            # final Bellman target
            y_js = rewards + (self.discount * next_q_values)

        loss = self.criterion(predicted_q, y_js)

        self.running_loss += loss.item()
        self.learned_counts += 1

        if done:
            episode_loss = self.running_loss / self.learned_counts
            self.loss_history.append(episode_loss)
            self.running_loss = 0
            self.learned_counts = 0

        self.optimizer.zero_grad()
        loss.backward() #TODO ?
        torch.nn.utils.clip_grad_norm_(self.main_network.parameters(), self.clip_grad_norm)
        self.optimizer.step()

    def hard_update(self):
        self.target_network.load_state_dict(self.main_network.state_dict())

    def update_epsilon(self, max_episodes):
        decay_amount = (self.epsilon_max - self.epsilon_min) / max_episodes
        self.epsilon_max = max(self.epsilon_min, self.epsilon_max - decay_amount)

    def save(self, path):
        torch.save(self.main_network.state_dict(), path)

In [ ]:
# Environment Wrappers
class observation_wrapper(gym.ObservationWrapper):
    """
    Wrapper class for modifying observations in the MountainCar-v0 environment.
    """

    def __init__(self, env):
        super().__init__(env)

        self.min_value = env.observation_space.low
        self.max_value = env.observation_space.high

    def observation(self, state):
        """
        Modifies the observation by clipping the values and normalizing it.
        """
        # Min-max normalization
        normalized_state = (state - self.min_value) / ( self.max_value - self.min_value )  

        return normalized_state

class reward_wrapper(gym.RewardWrapper):
    """
    Wrapper class for modifying rewards in the MountainCar-v0 environment.
    """

    def __init__(self, env):
        super().__init__(env)

    def reward(self, state):
        """
        Modifies the reward based on the current state of the environment.
        """
        # extract the position and current velocity based on the state
        current_position, current_velocity = state

        # Interpolate the value to the desired range (because the velocity normalized value would be in range of 0 to 1 and now it would be in range of -0.5 to 0.5)
        current_velocity = np.interp(
            current_velocity, np.array([0, 1]), np.array([-0.5, 0.5])
        )

        # Calculate the modified reward based on the current position and velocity of the car.
        degree = current_position * 360
        degree2radian = np.deg2rad(degree)
        modified_reward = 0.2 * (np.cos(degree2radian) + 2 * np.abs(current_velocity))

        # Step limitation
        # Subtract 0.5 to adjust the base reward (to limit useless steps).
        modified_reward -= (0.5)

        # Check if the car has surpassed a threshold of the path and is closer to the goal
        if current_position > 0.98:
            # Add a bonus reward (Reached the goal)
            modified_reward += 20 
            
        elif current_position > 0.92:
            # So close to the goal
            modified_reward += 10  
            
        elif current_position > 0.82:
            # car is closer to the goal
            modified_reward += 6  
            
        elif current_position > 0.65:
            # car is getting close. Thus, giving reward based on the position and the further it reached
            modified_reward += 1 - np.exp(-2 * current_position) 


        # Check if the car is coming down with velocity from left and goes with full velocity to right
        initial_position = 0.40842572  # Normalized value of initial position of the car which is extracted manually

        if current_velocity > 0.3 and current_position > initial_position + 0.1:
            # Add a bonus reward for this desired behavior
            modified_reward += (1 + 2 * current_position)  

        return modified_reward

class step_wrapper(gym.Wrapper):
    """
    A wrapper class for modifying the state and reward functions of the
    MountainCar-v0 environment.
    """

    def __init__(self, env):
        """
        Initializes the StepWrapper. This is the main class for wrapping the environment with it.
        """
        # We give the env here to initialize the gym.Wrapper superclass (inherited).
        super().__init__(env) 

        self.observation_wrapper = observation_wrapper(env)
        self.reward_wrapper = reward_wrapper(env)

    def step(self, action):
        """
        Executes a step in the environment with the provided action.The reason
        behind using this method is to have access to the state and reward functions return.
        """
        #TODO: DOes it enter here?
        state, reward, done, truncation, info = self.env.step(action) # Same as before as usual

        modified_state = self.observation_wrapper.observation(state)  # Give the state to another Wrapper, which returns a modified version of state
        
        modified_reward = self.reward_wrapper.reward(modified_state)  # Give the modified state to another Wrapper to return the modified reward

        # The same returns as usual but with modified versions of the state and reward functions
        return (
            modified_state,
            modified_reward,
            done,
            truncation,
            info,
        ) 

    def reset(self, seed):
        state, info = self.env.reset(seed=seed)  # Same as before as usual
        modified_state = self.observation_wrapper.observation(state)
        return (modified_state,info,)  

In [ ]:
# ==== Model Training & Testing ====
class Model_TrainTest:
    def __init__(self, hyperparams):

        # Define RL Hyperparameters
        self.train_mode = hyperparams["train_mode"]
        self.RL_load_path = hyperparams["RL_load_path"]
        self.save_path = hyperparams["save_path"]
        self.save_interval = hyperparams["save_interval"]

        self.clip_grad_norm = hyperparams["clip_grad_norm"]
        self.learning_rate = hyperparams["learning_rate"]
        self.discount_factor = hyperparams["discount_factor"]
        self.batch_size = hyperparams["batch_size"]
        self.update_frequency = hyperparams["update_frequency"]
        self.max_episodes = hyperparams["max_episodes"]
        self.max_steps = hyperparams["max_steps"]
        self.render = hyperparams["render"]

        self.epsilon_max = hyperparams["epsilon_max"]
        self.epsilon_min = hyperparams["epsilon_min"]
        self.epsilon_decay = hyperparams["epsilon_decay"]

        self.memory_capacity = hyperparams["memory_capacity"]

        self.render_fps = hyperparams["render_fps"]

        # Define Env
        self.env = gym.make("MountainCar-v0", max_episode_steps=self.max_steps, render_mode="human" if self.render else None)
        
        # For max frame rate make it 0
        self.env.metadata["render_fps"] = self.render_fps  

        # Ignore deprecated method warnings.
        # import warnings
        # warnings.filterwarnings("ignore", category=UserWarning)

        # Apply RewardWrapper
        self.env = step_wrapper(self.env)

        # Define the agent class
        self.agent = DQN_Agent(
            env=self.env,
            epsilon_max=self.epsilon_max,
            epsilon_min=self.epsilon_min,
            epsilon_decay=self.epsilon_decay,
            clip_grad_norm=self.clip_grad_norm,
            learning_rate=self.learning_rate,
            discount=self.discount_factor,
            memory_capacity=self.memory_capacity,
        )

    def train(self):
        total_steps = 0
        self.reward_history = []

        # Training loop over episodes
        for episode in range(1, self.max_episodes + 1):
            state, _ = self.env.reset(seed=seed)
            done = False
            truncation = False
            step_size = 0
            episode_reward = 0

            while not done and not truncation:
                action = self.agent.select_action(state)
                next_state, reward, done, truncation, _ = self.env.step(action)

                self.agent.replay_memory.store(state, action, next_state, reward, done)

                if len(self.agent.replay_memory) > self.batch_size:
                    self.agent.learn(self.batch_size, (done or truncation))

                    # Update target-network weights
                    if total_steps % self.update_frequency == 0:
                        self.agent.hard_update()

                state = next_state
                episode_reward += reward
                step_size += 1

            # Appends for tracking history
            self.reward_history.append(episode_reward)  # episode reward
            total_steps += step_size

            # Decay epsilon at the end of each episode
            self.agent.update_epsilon(self.max_episodes)

            # based on interval
            if episode % self.save_interval == 0:
                self.agent.save(self.save_path + "_" + f"{episode}" + ".pth")
                if episode != self.max_episodes:
                    self.plot_training(episode)
                print("\nModel saved\n")

            result = (
                f"Episode: {episode}, "
                # f"Total Steps: {total_steps}, "
                f"Ep Step: {step_size}, "
                f"Raw Reward: {episode_reward:.2f}, "
                f"Epsilon: {self.agent.epsilon_max:.2f}"
            )
            print(result)
        self.plot_training(episode)

    def test(self, max_episodes):
        # Load the weights of the test_network
        self.agent.main_network.load_state_dict(torch.load(self.RL_load_path))
        self.agent.main_network.eval()

        # Testing loop over episodes
        for episode in range(1, max_episodes + 1):
            state, _ = self.env.reset(seed=seed)
            done = False
            truncation = False
            step_size = 0
            episode_reward = 200

            while not done and not truncation:
                action = self.agent.select_action(state)
                next_state, reward, done, truncation, _ = self.env.step(action)

                state = next_state
                episode_reward += reward
                step_size += 1

            # Print log
            result = (
                f"Episode: {episode}, "
                f"Steps: {step_size:}, "
                f"Reward: {episode_reward:.2f}, "
            )
            print(result)

        pygame.quit()  # close the rendering window

    def plot_training(self, episode):
        # Calculate the Simple Moving Average (SMA) with a window size of 50
        sma = np.convolve(self.reward_history, np.ones(50) / 50, mode="valid")

        # Clip max (high) values for better plot analysis
        reward_history = np.clip(self.reward_history, a_min=None, a_max=100)
        sma = np.clip(sma, a_min=None, a_max=100)

        plt.figure()
        plt.title("Obtained Rewards")
        plt.plot(reward_history, label="Raw Reward", color="#4BA754", alpha=1)
        plt.plot(sma, label="Simple Moving Average 50", color="#F08100")
        plt.xlabel("Episode")
        plt.ylabel("Rewards")
        plt.legend()

        # Only save as file if last episode
        if episode == self.max_episodes:
            plt.savefig("./reward_plot.png", format="png", dpi=600, bbox_inches="tight")
        plt.tight_layout()
        plt.grid(True)
        plt.show()
        plt.clf()
        plt.close()

        plt.figure()
        plt.title("Network Loss")
        plt.plot(self.agent.loss_history, label="Loss", color="#8921BB", alpha=1)
        plt.xlabel("Episode")
        plt.ylabel("Loss")

        # Only save as file if last episode
        if episode == self.max_episodes:
            plt.savefig("./Loss_plot.png", format="png", dpi=600, bbox_inches="tight")
        plt.tight_layout()
        plt.grid(True)
        plt.show()


In [ ]:
# ==== Main Runner ====
if __name__ == "__main__":
    # Parameters:
    train_mode = False
    render = True
    RL_hyperparams = {
        "train_mode": train_mode,
        "RL_load_path": "./final_weights_1000.pth",
        "save_path": "./final_weights",
        "save_interval": 500,
        "clip_grad_norm": 5,
        "learning_rate": 1e-4,
        "discount_factor": 0.99,
        "batch_size": 64,
        "update_frequency": 20,
        "max_episodes": 1000 if train_mode else 10,
        "max_steps": 200,
        "render": render,
        
        "epsilon_max": 0.999 if train_mode else -1,
        "epsilon_min": 0.05,
        "epsilon_decay": 0.995,
        
        "memory_capacity": 50000 if train_mode else 0,
        "render_fps": 60,
    }

    # Run
    DRL = Model_TrainTest(RL_hyperparams)  # Define the instance
    # Train
    if train_mode:
        DRL.train()
    else:
        # Test
        DRL.test(max_episodes=RL_hyperparams["max_episodes"])
